# Phase 2 — Data Cleaning
Runs the data_cleaning + portfolio modules and displays results. All logic lives in `src/`, this notebook only calls it.

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd

from src.data_collection import config
from src.data_collection.saver import save_processed
from src.data_cleaning.alignment import align_trading_dates
from src.data_cleaning.cleaner import remove_stale_prices, winsorize_returns
from src.data_collection.returns import calculate_log_returns
from src.portfolio.construction import equal_weighted_portfolio_returns

In [2]:
# Load Phase 1 output (already-adjusted, already lightly-cleaned prices)
prices = pd.read_csv(config.DATA_PROCESSED_DIR / 'prices_adjusted_close.csv', index_col='date', parse_dates=True)
prices.tail()

,AAPL,MSFT,NVDA,JPM,XOM,AMZN,META,GOOGL,TSLA,UNH
date,,,,,,,,,,
2026-07-16,333.260010,401.100006,207.399994,343.149994,145.949997,249.889999,664.539978,354.459991,391.059998,423.380005
2026-07-17,333.739990,393.820007,202.809998,341.100006,147.360001,247.229996,646.010010,346.769989,380.839996,426.089996
2026-07-20,326.589996,402.290009,203.279999,338.869995,148.360001,249.990005,645.849976,351.989990,369.570007,421.549988
2026-07-21,327.739990,397.750000,207.289993,345.230011,151.710007,247.550003,643.809998,347.149994,378.929993,436.350006
2026-07-22,325.890015,390.339996,212.059998,348.209991,154.449997,244.850006,627.169983,342.089996,374.010010,431.309998


In [3]:
# Align trading dates across all tickers
aligned_prices, alignment_report = align_trading_dates(prices)
print(alignment_report)

AlignmentReport(dates_before=1393, dates_after=1393, dates_dropped=0, tickers_with_gaps_filled=[])


In [4]:
# Remove stale (repeated-run) prices
destaled_prices, stale_report = remove_stale_prices(aligned_prices, min_repeat_run=5)
print(stale_report)

StalePriceReport(stale_runs_found={}, values_replaced=0)


In [5]:
# Convert to log returns
log_returns = calculate_log_returns(destaled_prices)
log_returns.tail()

,AAPL,MSFT,NVDA,JPM,XOM,AMZN,META,GOOGL,TSLA,UNH
date,,,,,,,,,,
2026-07-16,0.017435,0.013731,-0.024293,-0.010898,0.009915,-0.020086,-0.024922,-0.045391,-0.008657,0.011545
2026-07-17,0.001439,-0.018317,-0.022380,-0.005992,0.009615,-0.010702,-0.028280,-0.021934,-0.026482,0.006380
2026-07-20,-0.021657,0.021279,0.002315,-0.006559,0.006763,0.011102,-0.000248,0.014941,-0.030039,-0.010712
2026-07-21,0.003515,-0.011350,0.019534,0.018594,0.022329,-0.009808,-0.003164,-0.013846,0.025011,0.034506
2026-07-22,-0.005661,-0.018806,0.022750,0.008595,0.017900,-0.010967,-0.026186,-0.014683,-0.013069,-0.011618


In [6]:
# Winsorize extreme return errors
winsorized_returns, winsorize_report = winsorize_returns(log_returns, lower_percentile=0.01, upper_percentile=0.99)
print(winsorize_report)

WinsorizeReport(lower_percentile=0.01, upper_percentile=0.99, values_clipped_low={'AAPL': 14, 'MSFT': 14, 'NVDA': 14, 'JPM': 14, 'XOM': 14, 'AMZN': 14, 'META': 14, 'GOOGL': 14, 'TSLA': 14, 'UNH': 14}, values_clipped_high={'AAPL': 14, 'MSFT': 14, 'NVDA': 14, 'JPM': 14, 'XOM': 14, 'AMZN': 14, 'META': 14, 'GOOGL': 14, 'TSLA': 14, 'UNH': 14})


In [7]:
# Build equal-weighted portfolio return series
portfolio_returns = equal_weighted_portfolio_returns(winsorized_returns)
portfolio_returns.tail()

date
2026-07-16   -0.008162
2026-07-17   -0.011665
2026-07-20   -0.001281
2026-07-21    0.008532
2026-07-22   -0.005174
Name: portfolio_return, dtype: float64

In [8]:
# Save Phase 2 outputs
save_processed(destaled_prices, 'prices_clean_phase2.csv', config.DATA_PROCESSED_DIR)
save_processed(winsorized_returns, 'returns_clean_winsorized.csv', config.DATA_PROCESSED_DIR)
save_processed(portfolio_returns.to_frame(), 'portfolio_returns_equal_weighted.csv', config.DATA_PROCESSED_DIR)
print('Phase 2 outputs saved.')

Phase 2 outputs saved.


In [9]:
# Rolling volatility series (required Phase 2 output)
rolling_volatility = winsorized_returns.rolling(window=21).std()
save_processed(rolling_volatility, 'rolling_volatility_21d.csv', config.DATA_PROCESSED_DIR)
rolling_volatility.tail()

,AAPL,MSFT,NVDA,JPM,XOM,AMZN,META,GOOGL,TSLA,UNH
date,,,,,,,,,,
2026-07-16,0.019477,0.022030,0.024118,0.015728,0.016040,0.021311,0.032562,0.022698,0.036391,0.016267
2026-07-17,0.019489,0.022153,0.024044,0.013966,0.016099,0.021457,0.033284,0.022920,0.036660,0.016144
2026-07-20,0.020078,0.020841,0.023885,0.014019,0.015974,0.019964,0.030495,0.022656,0.036943,0.015619
2026-07-21,0.020079,0.021049,0.023382,0.013180,0.015575,0.019192,0.030426,0.022605,0.037351,0.017107
2026-07-22,0.020134,0.020150,0.023830,0.012735,0.015845,0.015825,0.030557,0.020286,0.037275,0.017268
